<link rel="stylesheet" href="/site-assets/css/gemma.css">
<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Google+Symbols:opsz,wght,FILL,GRAD@20..48,100..700,0..1,-50..200" />

##### Copyright 2024 Google LLC。

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# 使用Keras產生PaliGemma輸出

<table class="tfo-notebook-buttons" align="left">
<td>
<a target="_blank" href="https://ai.google.dev/gemma/docs/paligemma/inference-with-keras"><img src="https://ai.google.dev/static/site-assets/images/docs/notebook-site-button.png" height="32" width="32" />在 ai.google.dev 查看</a>
</td>
<td>
<a target="_blank" href="https://colab.research.google.com/github/google-gemma/cookbook/blob/main/docs/paligemma/inference-with-keras.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />執行Google Colab</a>
</td>
<td> <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/google-gemma/cookbook/blob/main/docs/paligemma/inference-with-keras.ipynb"><img src="https://www.kaggle.com/static/images/logos/kaggle-logo-transparent-300.png" height="32" width="70"/>執行Kaggle</a>
</td>
<td> <a target="_blank" href="https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw.githubusercontent.com%2Fgoogle-gemma%2Fcookbook%2Fmain%2Fdocs%2Fpaligemma%2Finference-with-keras.ipynb"><img src="https://ai.google.dev/images/cloud-icon.svg" width="40" />在Vertex AI</a>開啟
</td>
<td>
<a target="_blank" href="https://github.com/google-gemma/cookbook/blob/main/docs/paligemma/inference-with-keras.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />查看GitHub</a>上的原始碼
</td>
</table>

PaliGemma 模型具有*多模式*功能，可讓您使用文字和圖像輸入資料產生輸出。您可以將圖像資料與這些模型結合使用，為您的請求提供額外的上下文，或使用模型來分析圖像的內容。本教學向您展示如何使用 PaliGemma 和 Keras 來分析圖像並回答有關圖像的問題。

## 這notebook裡有什麼

此notebook 使用PaliGemma 和Keras 並向您展示如何：
* 安裝 Keras 和所需的依賴項
* 下載 `PaliGemmaCausalLM`（用於因果視覺語言建模的預訓練 PaliGemma 變體），並使用它來創建模型
* 測試模型推斷有關所提供圖像的資訊的能力

## 開始之前

在閱讀此 notebook 之前，您應該熟悉 Python 程式碼，以及大型語言模型 (LLM) 的訓練方式。您不需要熟悉Keras，但有關Keras 的基本知識在閱讀範例程式碼時會很有幫助。

## 設定

以下部分介紹了讓notebook 使用PaliGemma 模型的初步步驟，包括模型存取、獲取API 金鑰以及設定notebook runtime。

### 訪問PaliGemma

在首次使用 PaliGemma 之前，您必須透過完成以下步驟透過 Kaggle 請求存取模型：
1. 登入 [Kaggle](https://www.kaggle.com)，或建立新的 Kaggle 帳戶（如果您還沒有帳戶）。
1. 前往 [PaliGemma 型號卡](https://www.kaggle.com/models/google/paligemma-2/) 並按一下 **請求存取**。
1. 填寫同意書並接受條款和條件。

### 設定您的 API 金鑰

若要使用 PaliGemma，您必須提供 Kaggle 使用者名稱和 Kaggle API 金鑰。
若要產生Kaggle API 金鑰，請開啟Kaggle 中的[**設定**頁面](https://www.kaggle.com/settings) 並點選**建立新 token**。這將觸發包含您的 API 憑證的 `kaggle.json` 檔案的下載。
然後，在 Colab 中，選擇左側窗格中的 **Secrets** (🔑) 並新增您的 Kaggle 使用者名稱和 Kaggle API 金鑰。將您的使用者名稱儲存在名稱`KAGGLE_USERNAME` 下，將您的API 金鑰儲存在名稱`KAGGLE_KEY` 下。

### 選擇runtime

要完成本教學，您需要擁有 Colab runtime 以及足夠的資源來執行 PaliGemma 模型。在這種情況下，您可以使用 T4 GPU：
1. 在 Colab 視窗的右上角，按一下 **▾（其他連線選項）** 下拉式選單。
1. 選擇**更改 runtime 類型**。
1. 在 **硬體加速器** 下，選擇 **T4 GPU**。

### 設定環境變數

設定`KAGGLE_USERNAME`、`KAGGLE_KEY` 和`KERAS_BACKEND` 的環境變數。

In [ ]:
import os
from google.colab import userdata

# Set up environmental variables
os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')
os.environ["KERAS_BACKEND"] = "jax"

### 安裝Keras

執行以下cell來安裝Keras。

In [ ]:
!pip install -U -q keras-nlp keras-hub kagglehub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 792.1/792.1 kB 9.2 MB/s eta 0:00:00


### 導入依賴並設定Keras

安裝此notebook所需的依賴項並設定Keras'後端。您也可以將Keras 設定為使用`bfloat16`，以便framework 使用更少的記憶體。

In [ ]:
import keras
import keras_hub
import numpy as np
import PIL
import requests
import io
import matplotlib
import re
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

keras.config.set_floatx("bfloat16")

## 載入模型

現在您已完成所有設置，您可以下載預先訓練的模型並建立一些實用方法來幫助您的模型產生響應。
在此步驟中，您使用`PaliGemmaCausalLM` 從Keras Hub 下載模型。此類幫助您管理和執行PaliGemma的因果視覺語言模型結構。 *因果視覺語言模型*根據先前的tokens預測下一個token。 Keras Hub 提供了許多流行的[模型架構](https://keras.io/keras_hub/api/models/) 的實作。

使用 `from_preset` 方法建立模型並列印其摘要。此過程大約需要一分鐘才能完成。

In [ ]:
paligemma = keras_hub.models.PaliGemmaCausalLM.from_preset("kaggle://keras/paligemma2/keras/pali_gemma2_mix_3b_224")
paligemma.summary()

## 建立實用方法

為了幫助您從模型產生回應，請建立兩個實用方法：
*   **`crop_and_resize`:** `read_img` 的輔助方法。此方法會裁切影像並將其大小調整為傳遞的大小，以便在不扭曲影像比例的情況下調整最終影像的大小。
*   **`read_img`:** `read_img_from_url` 的輔助方法。此方法實際上開啟圖像，調整其大小以使其適合模型的約束，並將其放入模型可以解釋的陣列中。
*   **`read_img_from_url`:** 透過有效的 URL 取得影像。您需要此方法將圖像傳遞給模型。

您將在此notebook 的下一步中使用`read_img_from_url`。

In [ ]:
def crop_and_resize(image, target_size):
    width, height = image.size
    source_size = min(image.size)
    left = width // 2 - source_size // 2
    top = height // 2 - source_size // 2
    right, bottom = left + source_size, top + source_size
    return image.resize(target_size, box=(left, top, right, bottom))

def read_image(url, target_size):
    contents = io.BytesIO(requests.get(url).content)
    image = PIL.Image.open(contents)
    image = crop_and_resize(image, target_size)
    image = np.array(image)
    # Remove alpha channel if necessary.
    if image.shape[2] == 4:
        image = image[:, :, :3]
    return image

def parse_bbox_and_labels(detokenized_output: str):
  matches = re.finditer(
      '<loc(?P<y0>\d\d\d\d)><loc(?P<x0>\d\d\d\d)><loc(?P<y1>\d\d\d\d)><loc(?P<x1>\d\d\d\d)>'
      ' (?P<label>.+?)( ;|$)',
      detokenized_output,
  )
  labels, boxes = [], []
  fmt = lambda x: float(x) / 1024.0
  for m in matches:
    d = m.groupdict()
    boxes.append([fmt(d['y0']), fmt(d['x0']), fmt(d['y1']), fmt(d['x1'])])
    labels.append(d['label'])
  return np.array(boxes), np.array(labels)

def display_boxes(image, boxes, labels, target_image_size):
  h, l = target_size
  fig, ax = plt.subplots()
  ax.imshow(image)
  for i in range(boxes.shape[0]):
      y, x, y2, x2 = (boxes[i]*h)
      width = x2 - x
      height = y2 - y
      # Create a Rectangle patch
      rect = patches.Rectangle((x, y),
                               width,
                               height,
                               linewidth=1,
                               edgecolor='r',
                               facecolor='none')
      # Add label
      plt.text(x, y, labels[i], color='red', fontsize=12)
      # Add the patch to the Axes
      ax.add_patch(rect)

  plt.show()

def display_segment_output(image, bounding_box, segment_mask, target_image_size):
    # Initialize a full mask with the target size
    full_mask = np.zeros(target_image_size, dtype=np.uint8)
    target_width, target_height = target_image_size

    for bbox, mask in zip(bounding_box, segment_mask):
        y1, x1, y2, x2 = bbox
        x1 = int(x1 * target_width)
        y1 = int(y1 * target_height)
        x2 = int(x2 * target_width)
        y2 = int(y2 * target_height)

        # Ensure mask is 2D before converting to Image
        if mask.ndim == 3:
            mask = mask.squeeze(axis=-1)
        mask = Image.fromarray(mask)
        mask = mask.resize((x2 - x1, y2 - y1), resample=Image.NEAREST)
        mask = np.array(mask)
        binary_mask = (mask > 0.5).astype(np.uint8)


        # Place the binary mask onto the full mask
        full_mask[y1:y2, x1:x2] = np.maximum(full_mask[y1:y2, x1:x2], binary_mask)
    cmap = plt.get_cmap('jet')
    colored_mask = cmap(full_mask / 1.0)
    colored_mask = (colored_mask[:, :, :3] * 255).astype(np.uint8)
    if isinstance(image, Image.Image):
        image = np.array(image)
    blended_image = image.copy()
    mask_indices = full_mask > 0
    alpha = 0.5

    for c in range(3):
        blended_image[:, :, c] = np.where(mask_indices,
                                          (1 - alpha) * image[:, :, c] + alpha * colored_mask[:, :, c],
                                          image[:, :, c])

    fig, ax = plt.subplots()
    ax.imshow(blended_image)
    plt.show()

## 產生輸出

載入模型並建立實用方法後，您可以使用圖像和文字資料prompt模型來產生回應。 PaliGemma 模型使用針對特定任務的特定prompt 語法進行訓練，例如`answer`、`caption` 和`detect`。有關PaliGemma prompt 任務語法的詳細信息，請參閱[PaliGemma prompt 和系統指令](https://ai.google.dev/gemma/docs/paligemma/prompt-system-instructions##prompt_task_syntax)。
透過使用以下程式碼將測試圖像載入到物件中，準備在生成 prompt 中使用的圖像：

In [ ]:
target_size = (224, 224)
image_url = 'https://storage.googleapis.com/keras-cv/models/paligemma/cow_beach_1.png'
cow_image = read_image(image_url, target_size)
matplotlib.pyplot.imshow(cow_image)

### 用特定語言回答

以下範例程式碼示範如何prompt PaliGemma 模型以取得有關所提供影像中出現的物件的資訊。此範例使用 `answer {lang}` 語法並顯示其他語言的其他問題：

In [ ]:
prompt = 'answer en where is the cow standing?\n'
# prompt = 'svar no hvor står kuen?\n'
# prompt = 'answer fr quelle couleur est le ciel?\n'
# prompt = 'responda pt qual a cor do animal?\n'

output = paligemma.generate(
    inputs={
        "images": cow_image,
        "prompts": prompt,
    }
)
print(output)

注意：使用 PaliGemma prompt 指令語法的提示必須以「`\n`」字元結尾。

### 使用 `detect` prompt

以下範例程式碼使用 `detect` prompt 語法來定位提供的圖像中的物件。程式碼使用先前定義的 `parse_bbox_and_labels()` 和 `display_boxes()` 函數來解釋模型輸出並顯示產生的邊界框。

In [ ]:
prompt = 'detect cow\n'
output = paligemma.generate(
    inputs={
        "images": cow_image,
        "prompts": prompt,
    }
)
boxes, labels = parse_bbox_and_labels(output)
display_boxes(cow_image, boxes, labels, target_size)

### 使用 `segment` prompt

以下範例程式碼使用 `segment` prompt 語法來定位物件佔據的影像區域。它使用 Google `big_vision` library 來解釋模型輸出並為分段物件產生遮罩。
在開始之前，請安裝 `big_vision` library 及其依賴項，如下列程式碼範例所示：

In [ ]:
import os
import sys

# TPUs with
if "COLAB_TPU_ADDR" in os.environ:
  raise "It seems you are using Colab with remote TPUs which is not supported."

# Fetch big_vision repository if python doesn't know about it and install
# dependencies needed for this notebook.
if not os.path.exists("big_vision_repo"):
  !git clone --quiet --branch=main --depth=1 \
     https://github.com/google-research/big_vision big_vision_repo

# Append big_vision code to python import path
if "big_vision_repo" not in sys.path:
  sys.path.append("big_vision_repo")


# Install missing dependencies. Assume jax~=0.4.25 with GPU available.
!pip3 install -q "overrides" "ml_collections" "einops~=0.7" "sentencepiece"

對於此分割範例，請載入並準備包含貓的不同圖像。

In [ ]:
cat = read_image('https://big-vision-paligemma.hf.space/file=examples/barsik.jpg', target_size)
matplotlib.pyplot.imshow(cat)

這是一個幫助解析 PaliGemma 的段輸出的函數

In [ ]:
import  big_vision.evaluators.proj.paligemma.transfers.segmentation as segeval
reconstruct_masks = segeval.get_reconstruct_masks('oi')
def parse_segments(detokenized_output: str) -> tuple[np.ndarray, np.ndarray]:
  matches = re.finditer(
      '<loc(?P<y0>\d\d\d\d)><loc(?P<x0>\d\d\d\d)><loc(?P<y1>\d\d\d\d)><loc(?P<x1>\d\d\d\d)>'
      + ''.join(f'<seg(?P<s{i}>\d\d\d)>' for i in range(16)),
      detokenized_output,
  )
  boxes, segs = [], []
  fmt_box = lambda x: float(x) / 1024.0
  for m in matches:
    d = m.groupdict()
    boxes.append([fmt_box(d['y0']), fmt_box(d['x0']), fmt_box(d['y1']), fmt_box(d['x1'])])
    segs.append([int(d[f's{i}']) for i in range(16)])
  return np.array(boxes), np.array(reconstruct_masks(np.array(segs)))

查詢PaliGemma分割影像中的貓

In [ ]:
prompt = 'segment cat\n'
output = paligemma.generate(
    inputs={
        "images": cat,
        "prompts": prompt,
    }
)

可視化PaliGemma產生的遮罩

In [ ]:
bboxes, seg_masks = parse_segments(output)
display_segment_output(cat, bboxes, seg_masks, target_size)

### 批次prompts

您可以在單一 prompt 中提供多個 prompt 指令作為一批指令。以下範例示範如何建立 prompt 文字以提供多個指令。
重要提示：每個 prompt 指令必須以「\n」字元結尾，如圖所示。

In [ ]:
prompts = [
    'answer en where is the cow standing?\n',
    'answer en what color is the cow?\n',
    'describe en\n',
    'detect cow\n',
    'segment cow\n',
]
images = [cow_image, cow_image, cow_image, cow_image, cow_image]
outputs = paligemma.generate(
    inputs={
        "images": images,
        "prompts": prompts,
    }
)
for output in outputs:
    print(output)